# Dephased-IC dataset generation

Generates the **normal** and **seizure** datasets on `single-knob-dephased-ic`.

Each recording warm-starts from a saved network state instead of
`h.finitialize(-65)`, which removes the initialization burst that fires at ~4.9 s
in every flagship recording. Everything else matches the flagship.

Run the cells in order. Output goes to
`notebooks/NEURON data parallel/dephased_ic/{normal,seizure}/`.

Method notes and the one known artifact (`DISCARD_EXTRA_MS`) are documented in
`analysis/dephase_generate.py`.


In [ ]:
# ==========================================================================
# CONFIG
# Normal vs seizure differ by ONE parameter: sahp_ainc_slow.
# Topology is not set here - this branch loads the flagship's already-built
# graph (926 neurons, 13356 edges) so the network is identical by construction.
# ==========================================================================
SAHP_NORMAL, SAHP_SEIZURE = 0.01, 0.004

CONFIG = {
    'build': dict(
        synapse_model='ampa_nmda', exc_tau=5.0, tau_nmda=350.0, nmda_ratio=3.0,
        exc_weight_scale=2.0, inh_weight_scale=2.5, depression_d=0.2, tau_d=500.0,
        noise_rate=5.0, noise_weight=0.007, adapt=True,
        gbar_kA_exc=0.006, gbar_kA_inh=0.004, tau_k=200.0,
        sahp_ainc_fast=0.005, sahp_tau_fast=300.0,
        sahp_ainc_slow=SAHP_NORMAL, sahp_tau_slow=6500.0, delay_per_distance=2.0,
    ),
    'sim': dict(dt=0.05, discard_transient_ms=1000.0),
}

STATES_TO_RUN   = ['normal', 'seizure']
N_RECORDINGS    = 50            # per state
N_WORKERS       = 5
RECORDING_MS    = 60000.0
NOISE_SEED_BASE = 1000          # Random123(base, gid, recording_index)
VOLTAGE         = 'probe'       # 'all' | 'probe' | 'none'
VOLTAGE_PROBE_N = 40
VOLTAGE_DT      = 5.0

# warm start: settle the network once, save states at these times, start
# recordings from them. DISCARD_EXTRA_MS drops the synaptic rebuild at sim ~1 s.
WARMUP_MS        = 130000.0
SNAPSHOT_TIMES   = [50000., 70000., 90000., 110000., 130000.]
DISCARD_EXTRA_MS = 3000.0

STATES = {'normal': SAHP_NORMAL, 'seizure': SAHP_SEIZURE}
print('%s | %d rec x %.0f s per state | %d workers | voltage=%s'
      % (STATES_TO_RUN, N_RECORDINGS, RECORDING_MS/1000, N_WORKERS, VOLTAGE))


## Preflight

In [ ]:
import os, sys, glob, json, subprocess, time, pickle
import numpy as np

REPO = os.path.abspath('..') if os.path.exists(os.path.join('..', 'neuron_simulation')) else os.path.abspath('.')
ANALYSIS = os.path.join(REPO, 'analysis')
DEPHASED = os.path.join(REPO, 'notebooks', 'NEURON data parallel', 'dephased_ic')
PY = sys.executable
for _p in (REPO, os.path.join(REPO, 'inference')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

def lib_path(s): return os.path.join(ANALYSIS, 'dephase_state_library_%s.npz' % s)
def out_dir(s):  return os.path.join(DEPHASED, s)

ok, need_library, todo = True, [], {}

if subprocess.run([PY, '-c', 'import neuron'], capture_output=True).returncode:
    ok = False
    print('FAIL: this kernel cannot import neuron - select the Python 3.9 interpreter')
if subprocess.run([PY, '-c', 'import sys; sys.path.insert(0, r"%s"); '
                  'from neuron_simulation.neurons import load_mechanisms; load_mechanisms()' % REPO],
                  capture_output=True, cwd=REPO).returncode:
    ok = False
    print('FAIL: mechanisms not built - run:  cd neuron_simulation && nrnivmodl mechanisms')

# does CONFIG['build'] still match the flagship?
_flag = pickle.load(open(os.path.join(REPO, 'notebooks', 'NEURON data parallel', 'normal',
                                      '20260721_163430', '_worker_config.pkl'), 'rb'))['build_kwargs']
_diff = {k: (_flag.get(k), v) for k, v in CONFIG['build'].items()
         if k != 'sahp_ainc_slow' and _flag.get(k) != v}
print("CONFIG['build']: %s" % ('matches the flagship' if not _diff else
      '%d change(s) -> %s' % (len(_diff), {k: '%r->%r' % v for k, v in _diff.items()})))

print('')
print('%-9s %-8s %-28s %s' % ('state', 'knob', 'library', 'recordings'))
for s in STATES_TO_RUN:
    lp = lib_path(s)
    if os.path.exists(lp):
        lb = np.load(lp)
        k = float(lb['sahp_ainc_slow']) if 'sahp_ainc_slow' in lb else None
        if k is not None and abs(k - STATES[s]) > 1e-12:
            txt, ok = 'MISMATCH: built at %.4f' % k, False
        else:
            txt = '%d snapshots' % lb['g_slow'].shape[0]
    else:
        txt = 'MISSING -> run the library cell'
        need_library.append(s)
    d = out_dir(s)
    todo[s] = [r for r in range(N_RECORDINGS)
               if not os.path.exists(os.path.join(d, 'recording%03d.npz' % r))]
    print('%-9s %-8.4f %-28s %d have / %d to go'
          % (s, STATES[s], txt, N_RECORDINGS - len(todo[s]), len(todo[s])))

_n = sum(len(v) for v in todo.values())
_min = (RECORDING_MS + 1000. + DISCARD_EXTRA_MS) / 1000. * 68.4 / 60.
print('')
print('generate %d recordings -> ~%.1f h with %d workers, ~%.2f GB'
      % (_n, _n * _min / 60. / max(1, N_WORKERS), N_WORKERS,
         _n * {'all': 77., 'probe': 3., 'none': 0.6}[VOLTAGE] / 1024.))
if need_library:
    print('build %d library(ies) first -> ~%.1f h (run concurrently)'
          % (len(need_library), (WARMUP_MS/1000.) * 68.4 / 3600.))
print('PREFLIGHT %s' % ('OK' if ok else 'FAILED'))


## Build warm-start libraries (once per state, ~2.5 h)

In [ ]:
if not need_library:
    print('all requested libraries already present - skip this cell')
else:
    snaps = [str(int(t)) for t in SNAPSHOT_TIMES]
    procs = []
    for s in need_library:
        log = os.path.join(ANALYSIS, '_dephase_lib_%s.log' % s)
        cmd = [PY, '-u', os.path.join(ANALYSIS, 'dephase_snapshot.py'),
               '--state', s, '--sahp-ainc-slow', str(STATES[s]),
               '--noise-seed-base', str(NOISE_SEED_BASE),
               '--build-overrides', json.dumps(CONFIG['build']),
               '--duration', str(WARMUP_MS), '--snapshots'] + snaps
        procs.append((s, subprocess.Popen(cmd, stdout=open(log, 'w'),
                                          stderr=subprocess.STDOUT), log))
        print('warming up %-8s (sahp_ainc_slow=%.4f) -> %s'
              % (s, STATES[s], os.path.basename(log)))
        time.sleep(15)

    t0 = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        print('[%5.1f min] warming up...' % ((time.time() - t0) / 60), flush=True)
        time.sleep(180)

    for s, p, log in procs:
        print('  %-8s exit=%s' % (s, p.returncode))
        if p.returncode:
            for line in open(log).read().strip().splitlines()[-6:]:
                print('     ' + line)

    print('')
    print('libraries now on disk:')
    for s in STATES_TO_RUN:
        lp = lib_path(s)
        if os.path.exists(lp):
            lb = np.load(lp)
            knob = float(lb['sahp_ainc_slow']) if 'sahp_ainc_slow' in lb else float('nan')
            print('  %-8s %d snapshots, knob %.4f, within-population sd %.5f uS'
                  % (s, lb['g_slow'].shape[0], knob, lb['g_slow'].std(axis=1).mean()))
        else:
            print('  %-8s STILL MISSING' % s)
    print('')
    print('re-run the preflight cell before generating.')

## Generate

In [ ]:
assert ok, 'preflight failed - do not run this cell'
assert not need_library, 'build the warm-start libraries first (cell 2b)'

# One worker slice per (state, chunk). Workers are shared across states, so
# N_WORKERS processes run at a time regardless of how many states are queued.
jobs = []
for s in STATES_TO_RUN:
    if not todo[s]:
        print('%-8s: nothing to do' % s)
        continue
    per = int(np.ceil(len(todo[s]) / max(1, N_WORKERS)))
    for w in range(N_WORKERS):
        chunk = todo[s][w*per:(w+1)*per]
        if chunk:
            jobs.append((s, w, chunk[0], len(chunk)))
print('\n%d worker jobs:' % len(jobs))
for s, w, start, count in jobs:
    print('  %-8s w%d: recordings %d..%d' % (s, w, start, start + count - 1))

running, done_jobs, t0 = [], [], time.time()
queue = list(jobs)
while queue or running:
    while queue and len(running) < N_WORKERS:
        s, w, start, count = queue.pop(0)
        log = os.path.join(ANALYSIS, '_dephase_nb_%s_w%d.log' % (s, w))
        cmd = [PY, '-u', os.path.join(ANALYSIS, 'dephase_generate.py'),
               '--state', s, '--sahp-ainc-slow', str(STATES[s]),
               '--noise-seed-base', str(NOISE_SEED_BASE),
               '--build-overrides', json.dumps(CONFIG['build']),
               '--start', str(start), '--count', str(count),
               '--duration', str(DURATION_MS),
               '--voltage', VOLTAGE,
               '--voltage-probe-n', str(VOLTAGE_PROBE_N),
               '--voltage-dt', str(VOLTAGE_DT),
               '--discard-extra-ms', str(DISCARD_EXTRA_MS)]
        running.append((s, w, subprocess.Popen(cmd, stdout=open(log, 'w'),
                                               stderr=subprocess.STDOUT), log))
        print('launched %-8s w%d (%d recordings)' % (s, w, count), flush=True)
        time.sleep(15)
    still = []
    for s, w, p, log in running:
        if p.poll() is None:
            still.append((s, w, p, log))
        else:
            done_jobs.append((s, w, p.returncode, log))
            print('  finished %-8s w%d exit=%s' % (s, w, p.returncode), flush=True)
    running = still
    if running or queue:
        have = {s: len(glob.glob(os.path.join(out_dir(s), 'recording*.npz')))
                for s in STATES_TO_RUN}
        print('[%5.1f min] %s | %d running, %d queued'
              % ((time.time()-t0)/60,
                 '  '.join('%s %d/%d' % (s, have[s], N_RECORDINGS) for s in STATES_TO_RUN),
                 len(running), len(queue)), flush=True)
        time.sleep(120)

print('\nall jobs exited after %.1f min' % ((time.time()-t0)/60))
bad = [(s, w, rc, log) for s, w, rc, log in done_jobs if rc]
for s, w, rc, log in bad:
    print('  FAILED %-8s w%d (exit %s):' % (s, w, rc))
    print('    ' + '\n    '.join(open(log).read().strip().splitlines()[-6:]))
for s in STATES_TO_RUN:
    d = out_dir(s)
    print('%-8s: %d recordings, %d rasters, %d summaries'
          % (s, len(glob.glob(os.path.join(d, 'recording*.npz'))),
             len(glob.glob(os.path.join(d, 'recording*_raster*.png'))),
             len(glob.glob(os.path.join(d, '_summary_*.json')))))
if bad:
    print('\n%d job(s) failed - check the logs above before using the dataset.' % len(bad))

## Validate

In [ ]:
for s in STATES_TO_RUN:
    print('=' * 72)
    print('VALIDATING state: %s  (sahp_ainc_slow = %.4f)' % (s, STATES[s]))
    print('=' * 72)
    r = subprocess.run([PY, '-u', os.path.join(ANALYSIS, 'dephase_validate.py'),
                        '--state', s], capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print('STDERR:', r.stderr[-1500:])
    print()